# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds one transparent, deterministic baseline action score for the FlyRank starter dataset. It first audits two signals, then ranks content using only decision-time fields, and finally reviews the top 20 with a skeptical eye.

**Important:** `trend_direction` / `is_declining_label` is used only for signal auditing and post-score evaluation. It is never used to calculate the baseline score, reason code, or action label.

## 0. Load the starter data

The repo ships `data/raw/content_refresh_anonymized.csv` with 30,000 content items. The data dictionary defines `trend_direction` as the source of the declining label, so it is excluded from scoring.

In [10]:
!git clone https://github.com/ShahvezAli784/-flyrank-machine-learning

Cloning into '-flyrank-machine-learning'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 132 (delta 46), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.85 MiB | 13.53 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [12]:
%cd /content/-flyrank-machine-learning

/content/-flyrank-machine-learning


In [15]:
from pathlib import Path
import pandas as pd

# Find the repository root relative to this notebook
REPO_ROOT = Path.cwd()

# If running from work/notebooks, move up two levels
if (REPO_ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
elif (REPO_ROOT.parent.parent / "data/raw/content_refresh_anonymized.csv").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
    DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
else:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from the FlyRank repository."
    )

df = pd.read_csv(DATA_PATH)

print("Repository root:", REPO_ROOT)
print("Data path:", DATA_PATH)
print("Shape:", df.shape)

Repository root: /content/-flyrank-machine-learning
Data path: /content/-flyrank-machine-learning/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)


## 1. Signal checks

### Signal 1 — Staleness

This is linked to the session's refresh-flag logic: older content is a natural candidate for refresh review. I bucket `days_since_last_update` and inspect the observed declining rate in each bucket.

**Verdict: MIXED.** The 91–180 day bucket has a higher observed decline rate than the 0–30 day bucket, but the 181+ bucket drops again. Staleness is therefore directionally useful, but not monotonic enough to call it fully confirmed.

In [17]:
# Audit-only label
# Used only to evaluate the signal, NOT for scoring.

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Audit label created.")
print(df["is_declining_label"].value_counts())
print(f"Declining rate: {df['is_declining_label'].mean():.3%}")

Audit label created.
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Declining rate: 54.207%


In [19]:
stale_bins = [-np.inf, 30, 90, 180, np.inf]

stale_labels = ["0-30", "31-90", "91-180", "181+"]

stale_audit = df.assign(
    bucket=pd.cut(
        df["days_since_last_update"],
        bins=stale_bins,
        labels=stale_labels
    )
).groupby("bucket", observed=False).agg(
    n=("content_id", "size"),
    decline_rate=("is_declining_label", "mean"),
)

stale_audit["decline_rate"] = stale_audit["decline_rate"].map(
    lambda x: f"{x:.3%}"
)

print(stale_audit.to_string())

print("Verdict: MIXED")

            n decline_rate
bucket                    
0-30    20480      51.138%
31-90     175      58.857%
91-180   9171      61.106%
181+      174      47.126%
Verdict: MIXED


### Signal 2 — Visibility / volume

This is linked to the session's quick-win logic: a page with meaningful search volume has more observable opportunity than a page with almost no impressions. I bucket 90-day impressions and inspect the observed declining rate.

**Verdict: MIXED.** Decline rate is much higher above the 0–99 impression bucket, but the highest-volume 30,000+ bucket declines again. Volume is useful for prioritization, but it is not a simple monotonic risk signal.

In [20]:
volume_bins = [-np.inf, 99, 499, 2999, 29999, np.inf]
volume_labels = ["0-99", "100-499", "500-2999", "3000-29999", "30000+"]
volume_audit = df.assign(
    bucket=pd.cut(df["impressions_90d"], bins=volume_bins, labels=volume_labels)
).groupby("bucket", observed=False).agg(
    n=("content_id", "size"),
    decline_rate=("is_declining_label", "mean"),
)
volume_audit["decline_rate"] = volume_audit["decline_rate"].map(lambda x: f"{x:.3%}")
print(volume_audit.to_string())
print("Verdict: MIXED")


               n decline_rate
bucket                       
0-99        7994      38.904%
100-499     5280      60.436%
500-2999    8443      62.063%
3000-29999  7205      58.612%
30000+      1078      46.197%
Verdict: MIXED


## 2. My one transparent baseline rule

**Plain-language rule:** Prioritize pages that are both **stale (180+ days since update)** and **visible (500+ impressions in the last 90 days)**. Within that priority group, use simple percentile ranks of visibility and staleness to order the queue. Pages that are not both stale and visible remain lower-priority monitoring/review items.

**Score:**

`score = 2 × stale_visible + 0.5 × visibility_percentile + 0.5 × freshness_risk_percentile`

There are no fitted weights and no future-window inputs.

**Reason code:** exactly one of `stale_visible_page`, `stale_page`, `visible_page`, or `monitor`.

**Action label:** `refresh` for stale + visible pages, `review` for stale pages without enough visibility, otherwise `monitor`.

In [22]:
from pathlib import Path

OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

print("Output path:", OUTPUT_PATH)

Output path: /content/-flyrank-machine-learning/work/outputs/baseline_action_score.csv


In [23]:
# Decision-time inputs only. The audit label and trend_direction are deliberately excluded.
work = df.copy()

work["visibility_percentile"] = np.log1p(work["impressions_90d"]).rank(method="average", pct=True)
work["freshness_risk_percentile"] = work["days_since_last_update"].rank(method="average", pct=True)
work["stale_flag"] = (work["days_since_last_update"] >= 180).astype(int)
work["visible_flag"] = (work["impressions_90d"] >= 500).astype(int)
work["stale_visible_flag"] = work["stale_flag"] * work["visible_flag"]

work["score"] = (
    2.0 * work["stale_visible_flag"]
    + 0.5 * work["visibility_percentile"]
    + 0.5 * work["freshness_risk_percentile"]
)

work["reason_code"] = np.select(
    [
        work["stale_visible_flag"].eq(1),
        work["stale_flag"].eq(1),
        work["visible_flag"].eq(1),
    ],
    ["stale_visible_page", "stale_page", "visible_page"],
    default="monitor",
)

work["action_label"] = np.select(
    [work["stale_visible_flag"].eq(1), work["stale_flag"].eq(1)],
    ["refresh", "review"],
    default="monitor",
)

work = work.sort_values(["score", "impressions_90d", "days_since_last_update"], ascending=[False, False, False]).reset_index(drop=True)
work["baseline_rank"] = np.arange(1, len(work) + 1)

queue_columns = [
    "content_id", "baseline_rank", "score", "reason_code", "action_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_age_days"
]
queue = work[queue_columns].copy()
queue.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote: {OUTPUT_PATH}")
print(f"Rows ranked: {len(queue):,}")
print("Action mix:")
print(queue["action_label"].value_counts().to_string())
print("Reason mix:")
print(queue["reason_code"].value_counts().to_string())


Wrote: /content/-flyrank-machine-learning/work/outputs/baseline_action_score.csv
Rows ranked: 30,000
Action mix:
action_label
monitor    29826
review       157
refresh       17
Reason mix:
reason_code
visible_page          16709
monitor               13117
stale_page              157
stale_visible_page       17


## 2b. Baseline evaluation (audit only)

For the ranking metric, the existing `is_declining_label` is used **after** scoring to measure precision@K. It does not enter the score or action logic. This is a same-slice descriptive baseline check, not a held-out model evaluation.

In [24]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
print(f"Base declining rate: {base_rate:.3%}")
for k in [10, 20, 50]:
    p_at_k = precision_at_k(work["score"], work["is_declining_label"], k)
    print(f"Precision@{k}: {p_at_k:.3%}  (lift vs base rate: {p_at_k/base_rate:.2f}x)")

metrics = {
    "rows_ranked": int(len(work)),
    "base_declining_rate": float(base_rate),
    "precision_at_10": float(precision_at_k(work["score"], work["is_declining_label"], 10)),
    "precision_at_20": float(precision_at_k(work["score"], work["is_declining_label"], 20)),
    "precision_at_50": float(precision_at_k(work["score"], work["is_declining_label"], 50)),
    "rule": "2 * stale_visible + 0.5 * visibility_percentile + 0.5 * freshness_risk_percentile",
    "signal_verdicts": {"staleness": "MIXED", "visibility_volume": "MIXED"},
}
with open(OUTPUT_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote: {OUTPUT_DIR / 'baseline_metrics.json'}")


Base declining rate: 54.207%
Precision@10: 100.000%  (lift vs base rate: 1.84x)
Precision@20: 90.000%  (lift vs base rate: 1.66x)
Precision@50: 76.000%  (lift vs base rate: 1.40x)
Wrote: /content/-flyrank-machine-learning/work/outputs/baseline_metrics.json


## 3. Top-20 review

Each row gets an action, its single reason code, a confidence note, and one concrete condition that would make the pick wrong. The point is to look for weak logic rather than assume the ranking is correct.

In [25]:
top20 = work.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "stale_visible_page":
        return "Higher confidence: it crosses both explicit baseline thresholds."
    if row["reason_code"] == "stale_page":
        return "Moderate confidence: stale, but visibility is below the priority threshold."
    if row["reason_code"] == "visible_page":
        return "Lower confidence: visible, but not stale enough for the refresh rule."
    return "Lower confidence: no primary baseline trigger."

def wrong_if(row):
    if row["reason_code"] == "stale_visible_page":
        return "Wrong if the old update date reflects an intentional evergreen page or if current demand is temporary/unrepresentative."
    if row["reason_code"] == "stale_page":
        return "Wrong if the page is intentionally left unchanged and has too little demand to justify refresh work."
    if row["reason_code"] == "visible_page":
        return "Wrong if the page is already healthy and needs no refresh despite having high visibility."
    return "Wrong if an unmeasured issue makes this page important despite no baseline trigger."

review = top20[["baseline_rank", "content_id", "action_label", "reason_code", "score", "impressions_90d", "days_since_last_update", "avg_position", "ctr"]].copy()
review["confidence_note"] = review.apply(confidence_note, axis=1)
review["what_would_make_it_wrong"] = review.apply(wrong_if, axis=1)

for _, r in review.iterrows():
    print(f"#{int(r.baseline_rank)} {r.content_id} | action={r.action_label} | reason={r.reason_code} | score={r.score:.4f}")
    print(f"  Why: {r.confidence_note}")
    print(f"  What would make it wrong: {r.what_would_make_it_wrong}")


#1 content_cf56e2e2e282 | action=refresh | reason=stale_visible_page | score=2.9913
  Why: Higher confidence: it crosses both explicit baseline thresholds.
  What would make it wrong: Wrong if the old update date reflects an intentional evergreen page or if current demand is temporary/unrepresentative.
#2 content_7368877ea310 | action=refresh | reason=stale_visible_page | score=2.9911
  Why: Higher confidence: it crosses both explicit baseline thresholds.
  What would make it wrong: Wrong if the old update date reflects an intentional evergreen page or if current demand is temporary/unrepresentative.
#3 content_1bfaa38ff26c | action=refresh | reason=stale_visible_page | score=2.9758
  Why: Higher confidence: it crosses both explicit baseline thresholds.
  What would make it wrong: Wrong if the old update date reflects an intentional evergreen page or if current demand is temporary/unrepresentative.
#4 content_0a91db491d14 | action=refresh | reason=stale_visible_page | score=2.9520
  Wh

## 4. Weak picks + leakage check

The top-20 review should find at least one weak pick. Here I flag top-20 rows that are not actually declining according to the audit label. Those are **review findings**, not inputs to the rule.

The leakage check confirms that the score was built without `trend_direction`, `trend_pct`, or the audit label.

In [26]:
weak = work.head(20).loc[work.head(20)["is_declining_label"].eq(0), ["baseline_rank", "content_id", "reason_code", "action_label", "impressions_90d", "days_since_last_update", "trend_direction"]]
print("Weak picks in top 20 (non-declining audit rows):")
print(weak.to_string(index=False))

score_inputs = {
    "impressions_90d", "days_since_last_update"
}
forbidden_inputs = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"
}
print("\nLeakage check:")
print("- Score inputs:", sorted(score_inputs))
print("- Forbidden future/label fields excluded:", sorted(forbidden_inputs))
print("- Label-derived fields used in score:", sorted(score_inputs & forbidden_inputs))
assert not (score_inputs & forbidden_inputs)
print("PASS: no future-window or label-derived input is used by the baseline score.")


Weak picks in top 20 (non-declining audit rows):
 baseline_rank           content_id        reason_code action_label  impressions_90d  days_since_last_update trend_direction
            12 content_bdbec75c1148 stale_visible_page      refresh             1316                     194          stable
            18 content_a5dbb404bdc2       visible_page      monitor            79035                     106          stable

Leakage check:
- Score inputs: ['days_since_last_update', 'impressions_90d']
- Forbidden future/label fields excluded: ['clicks_last_30d', 'impressions_last_30d', 'is_declining_label', 'sessions_last_30d', 'trend_direction', 'trend_pct']
- Label-derived fields used in score: []
PASS: no future-window or label-derived input is used by the baseline score.


## Self-check

- [x] Two signal checks are present, each with a visible bucket table and `n`.
- [x] At least one audited signal is linked to the session's refresh/flag logic (staleness).
- [x] Each signal has one verdict: `MIXED`.
- [x] One transparent rule is stated in plain language before coding.
- [x] Each ranked row has exactly one reason code and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Top-20 review includes action, reason, confidence, and what would make it wrong.
- [x] At least one weak pick is surfaced.
- [x] Future-window and label-derived inputs are excluded from the score.
- [x] The CSV is an output artifact and should remain untracked; the notebook regenerates it.
